# Prog09: Autoassociative and Heteroassociative Memory Networks


In [24]:
import numpy as np
import matplotlib.pyplot as plt


# Define binary patterns


In [25]:
# Define binary patterns (using -1 and 1 for bipolar representation)
patterns = {
    'A': np.array([-1, 1, -1, 1, -1, 1, -1, 1, -1]),
    'B': np.array([1, -1, 1, -1, 1, -1, 1, -1, 1]),
    'C': np.array([1, 1, -1, -1, 1, 1, -1, -1, 1]),
    'D': np.array([-1, -1, 1, 1, -1, -1, 1, 1, -1])
}

# Heteroassociative patterns (input -> output mappings)
hetero_patterns = {
    'A': (np.array([-1, 1, -1, 1, -1]), np.array([1, -1, 1])),
    'B': (np.array([1, -1, 1, -1, 1]), np.array([-1, 1, -1])),
    'C': (np.array([1, 1, -1, -1, 1]), np.array([1, 1, -1])),
    'D': (np.array([-1, -1, 1, 1, -1]), np.array([-1, -1, 1]))
}

print("Autoassociative patterns:")
for name, pattern in patterns.items():
    print(f"{name}: {pattern}")

print("\nHeteroassociative patterns:")
for name, (input_pat, output_pat) in hetero_patterns.items():
    print(f"{name}: {input_pat} -> {output_pat}")


Autoassociative patterns:
A: [-1  1 -1  1 -1  1 -1  1 -1]
B: [ 1 -1  1 -1  1 -1  1 -1  1]
C: [ 1  1 -1 -1  1  1 -1 -1  1]
D: [-1 -1  1  1 -1 -1  1  1 -1]

Heteroassociative patterns:
A: [-1  1 -1  1 -1] -> [ 1 -1  1]
B: [ 1 -1  1 -1  1] -> [-1  1 -1]
C: [ 1  1 -1 -1  1] -> [ 1  1 -1]
D: [-1 -1  1  1 -1] -> [-1 -1  1]


# Autoassociative Memory Implementation


In [26]:
class AutoassociativeMemory:
    def __init__(self, pattern_size):
        self.pattern_size = pattern_size
        self.weight_matrix = np.zeros((pattern_size, pattern_size))
        
    def store_patterns(self, patterns):
        """Store patterns using Hebbian learning"""
        for pattern in patterns:
            # Outer product for weight matrix update
            self.weight_matrix += np.outer(pattern, pattern)
        
        # Set diagonal to zero (no self-connections)
        np.fill_diagonal(self.weight_matrix, 0)
        
    def recall(self, input_pattern, max_iterations=10):
        """Recall pattern using iterative update"""
        current_pattern = input_pattern.copy()
        
        for iteration in range(max_iterations):
            # Update each neuron
            new_pattern = np.sign(np.dot(self.weight_matrix, current_pattern))
            
            # Check for convergence
            if np.array_equal(new_pattern, current_pattern):
                break
                
            current_pattern = new_pattern
            
        return current_pattern, iteration + 1
    
    def get_energy(self, pattern):
        """Calculate energy of the pattern"""
        return -0.5 * np.dot(pattern, np.dot(self.weight_matrix, pattern))


# Train Autoassociative Memory


In [27]:
# Create and train autoassociative memory
auto_memory = AutoassociativeMemory(pattern_size=9)
pattern_list = list(patterns.values())
auto_memory.store_patterns(pattern_list)

print("Autoassociative Memory trained!")
print(f"Weight matrix shape: {auto_memory.weight_matrix.shape}")
print(f"Weight matrix:\n{auto_memory.weight_matrix}")


Autoassociative Memory trained!
Weight matrix shape: (9, 9)
Weight matrix:
[[ 0.  0.  0. -4.  4.  0.  0. -4.  4.]
 [ 0.  0. -4.  0.  0.  4. -4.  0.  0.]
 [ 0. -4.  0.  0.  0. -4.  4.  0.  0.]
 [-4.  0.  0.  0. -4.  0.  0.  4. -4.]
 [ 4.  0.  0. -4.  0.  0.  0. -4.  4.]
 [ 0.  4. -4.  0.  0.  0. -4.  0.  0.]
 [ 0. -4.  4.  0.  0. -4.  0.  0.  0.]
 [-4.  0.  0.  4. -4.  0.  0.  0. -4.]
 [ 4.  0.  0. -4.  4.  0.  0. -4.  0.]]


# Test Autoassociative Memory with noisy patterns


In [28]:
# Test with noisy versions of stored patterns
def add_noise(pattern, noise_level=0.3):
    """Add random noise to a pattern"""
    noisy = pattern.copy()
    n_noise = int(len(pattern) * noise_level)
    noise_indices = np.random.choice(len(pattern), n_noise, replace=False)
    noisy[noise_indices] *= -1  # Flip bits
    return noisy

print("Testing Autoassociative Memory:")
for name, original in patterns.items():
    # Add noise
    noisy = add_noise(original, noise_level=0.3)
    
    # Recall
    recalled, iterations = auto_memory.recall(noisy)
    
    # Check if recall is correct
    is_correct = np.array_equal(recalled, original)
    
    print(f"\nPattern {name}:")
    print(f"  Original:  {original}")
    print(f"  Noisy:     {noisy}")
    print(f"  Recalled:  {recalled}")
    print(f"  Correct:   {is_correct} (iterations: {iterations})")
    print(f"  Energy:    {auto_memory.get_energy(recalled):.2f}")


Testing Autoassociative Memory:

Pattern A:
  Original:  [-1  1 -1  1 -1  1 -1  1 -1]
  Noisy:     [-1  1 -1 -1 -1  1 -1  1  1]
  Recalled:  [-1.  1. -1.  1. -1.  1. -1.  1. -1.]
  Correct:   True (iterations: 3)
  Energy:    -64.00

Pattern B:
  Original:  [ 1 -1  1 -1  1 -1  1 -1  1]
  Noisy:     [ 1  1  1 -1  1  1  1 -1  1]
  Recalled:  [ 1.  1.  1. -1.  1.  1.  1. -1.  1.]
  Correct:   False (iterations: 10)
  Energy:    -32.00

Pattern C:
  Original:  [ 1  1 -1 -1  1  1 -1 -1  1]
  Noisy:     [ 1  1 -1 -1  1  1 -1  1 -1]
  Recalled:  [ 1.  1. -1. -1.  1.  1. -1. -1.  1.]
  Correct:   True (iterations: 3)
  Energy:    -64.00

Pattern D:
  Original:  [-1 -1  1  1 -1 -1  1  1 -1]
  Noisy:     [-1 -1  1  1 -1  1 -1  1 -1]
  Recalled:  [-1. -1.  1.  1. -1.  1. -1.  1. -1.]
  Correct:   False (iterations: 10)
  Energy:    -32.00


# Heteroassociative Memory Implementation


In [29]:
class HeteroassociativeMemory:
    def __init__(self, input_size, output_size):
        self.input_size = input_size
        self.output_size = output_size
        self.weight_matrix = np.zeros((output_size, input_size))
        
    def store_patterns(self, input_patterns, output_patterns):
        """Store input-output pattern pairs using Hebbian learning"""
        for input_pat, output_pat in zip(input_patterns, output_patterns):
            # Outer product for weight matrix update
            self.weight_matrix += np.outer(output_pat, input_pat)
    
    def recall(self, input_pattern):
        """Recall output pattern from input"""
        # Compute output
        output = np.sign(np.dot(self.weight_matrix, input_pattern))
        return output
    
    def get_energy(self, input_pattern, output_pattern):
        """Calculate energy of the input-output pair"""
        return -np.dot(output_pattern, np.dot(self.weight_matrix, input_pattern))


# Train Heteroassociative Memory


In [30]:
# Create and train heteroassociative memory
input_patterns = [pair[0] for pair in hetero_patterns.values()]
output_patterns = [pair[1] for pair in hetero_patterns.values()]

hetero_memory = HeteroassociativeMemory(input_size=5, output_size=3)
hetero_memory.store_patterns(input_patterns, output_patterns)

print("Heteroassociative Memory trained!")
print(f"Weight matrix shape: {hetero_memory.weight_matrix.shape}")
print(f"Weight matrix:\n{hetero_memory.weight_matrix}")


Heteroassociative Memory trained!
Weight matrix shape: (3, 5)
Weight matrix:
[[ 0.  4. -4.  0.  0.]
 [ 4.  0.  0. -4.  4.]
 [-4.  0.  0.  4. -4.]]


# Test Heteroassociative Memory


In [31]:
print("Testing Heteroassociative Memory:")
for name, (input_pat, expected_output) in hetero_patterns.items():
    # Test with original input
    recalled_output = hetero_memory.recall(input_pat)
    is_correct = np.array_equal(recalled_output, expected_output)
    
    print(f"\nPattern {name}:")
    print(f"  Input:     {input_pat}")
    print(f"  Expected:  {expected_output}")
    print(f"  Recalled:  {recalled_output}")
    print(f"  Correct:   {is_correct}")
    print(f"  Energy:    {hetero_memory.get_energy(input_pat, recalled_output):.2f}")
    
    # Test with noisy input
    noisy_input = add_noise(input_pat, noise_level=0.2)
    noisy_recalled = hetero_memory.recall(noisy_input)
    noisy_correct = np.array_equal(noisy_recalled, expected_output)
    
    print(f"  Noisy Input: {noisy_input}")
    print(f"  Noisy Recall: {noisy_recalled}")
    print(f"  Noisy Correct: {noisy_correct}")


Testing Heteroassociative Memory:

Pattern A:
  Input:     [-1  1 -1  1 -1]
  Expected:  [ 1 -1  1]
  Recalled:  [ 1. -1.  1.]
  Correct:   True
  Energy:    -32.00
  Noisy Input: [-1  1 -1 -1 -1]
  Noisy Recall: [ 1. -1.  1.]
  Noisy Correct: True

Pattern B:
  Input:     [ 1 -1  1 -1  1]
  Expected:  [-1  1 -1]
  Recalled:  [-1.  1. -1.]
  Correct:   True
  Energy:    -32.00
  Noisy Input: [-1 -1  1 -1  1]
  Noisy Recall: [-1.  1. -1.]
  Noisy Correct: True

Pattern C:
  Input:     [ 1  1 -1 -1  1]
  Expected:  [ 1  1 -1]
  Recalled:  [ 1.  1. -1.]
  Correct:   True
  Energy:    -32.00
  Noisy Input: [ 1  1  1 -1  1]
  Noisy Recall: [ 0.  1. -1.]
  Noisy Correct: False

Pattern D:
  Input:     [-1 -1  1  1 -1]
  Expected:  [-1 -1  1]
  Recalled:  [-1. -1.  1.]
  Correct:   True
  Energy:    -32.00
  Noisy Input: [-1 -1  1  1  1]
  Noisy Recall: [-1. -1.  1.]
  Noisy Correct: True
